# 05 · SQL analytics and reconciliation

Actual Superstore sample data, not simulated outputs. Run cells from top to bottom. Dollar amounts are USD.


In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/project.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the extracted project folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.project import load, summarize, save_table, chart
pd.set_option('display.max_columns',30)
pd.set_option('display.width',160)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(10,5),'axes.spines.top':False,'axes.spines.right':False})

SQLite runs locally using Python's standard library; no server or credentials are needed. This database is regenerated from the clean CSV. SQL monthly LAG assumes a complete monthly series, explicitly checked below.

In [2]:
import sqlite3
import re
df=load()
months=df['Order Date'].dt.to_period('M').nunique()
assert months==len(pd.period_range(df['Order Date'].min(),df['Order Date'].max(),freq='M'))
con=sqlite3.connect(ROOT/'data/processed/sales.sqlite')
df.to_sql('sales',con,if_exists='replace',index=False)
sql=(ROOT/'sql/analysis.sql').read_text()
results={}
for name,query in re.findall(r'-- query: (\w+)\n(.*?)(?=-- query:|\Z)',sql,re.S):
    results[name]=pd.read_sql_query(query,con)
    print('\n',name,'\n',results[name].head(12).to_string(index=False))
    results[name].to_csv(ROOT/'reports/tables'/f'sql_{name}.csv',index=False)
assert np.isclose(results['kpis'].iloc[0]['Sales'],df.Sales.sum())
assert np.isclose(results['kpis'].iloc[0]['Profit'],df.Profit.sum())
assert int(results['kpis'].iloc[0]['Orders'])==df['Order ID'].nunique()
con.close()
print('SQL and pandas totals reconcile.')


 kpis 
        Sales      Profit  Orders  Customers  Margin_pct        AOV
2297200.8603 286397.0217    5009        793   12.467217 458.614666

 category 
        Category       Sales      Profit  Margin_pct
     Technology 836154.0330 145454.9481   17.395712
      Furniture 741999.7953  18451.2728    2.486695
Office Supplies 719047.0320 122490.8008   17.035158

 loss_states 
          State       Sales      Profit
         Texas 170188.0458 -25729.3563
          Ohio  78258.1360 -16971.3766
  Pennsylvania 116511.9140 -15559.9603
      Illinois  80166.1010 -12607.8870
North Carolina  55603.1640  -7490.9122
      Colorado  32108.1180  -6527.8579
     Tennessee  30661.8730  -5341.6936
       Arizona  35282.0010  -3427.9246
       Florida  89473.7080  -3399.3017
        Oregon  17431.1500  -1190.4705

 monthly_yoy 
   Month      Sales  Prior_Year_Sales  YoY_pct
2014-01 14236.8950               NaN      NaN
2014-02  4519.8920               NaN      NaN
2014-03 55691.0090               NaN 